# Procedural Frozen Lake — tutorial rollout

This notebook walks through **`Procedural-FrozenLake-v1`**: a Gymnasium Frozen Lake variant that generates valid maps on a fixed canvas, supports jagged lake shorelines bounded by trees, mirror ice, sleigh warps, fog-of-war rendering, and optimal Q-values in `info`.

We will:

1. Import the package and register the environment
2. Build an env with rendering, fog, variable lake shape, multiple starts and goals (with per-goal rewards), and Q\* labels
3. Print a **readable text map** with row/column labels and a tile legend
4. Run **20 episodes** on the **same map**, ramping from a random policy to a pure Q\* greedy policy
5. Play back the captured frames as an embedded video

Fog-of-war tiles stay revealed across episode resets; unvisited trees also start as `?` and are revealed when you walk into them. Only `regenerate_map=True` clears exploration.

## Imports and settings

Import `procedural_frozenlake` first — that registers `Procedural-FrozenLake-v1` with Gymnasium.

Frozen Lake renders through **pygame**. The `SDL_*` environment variables let `rgb_array` rendering work in headless notebook servers (no physical display required).

Key settings:

- `NUM_EPISODES = 20` — how many episodes to record
- `REGENERATE_MAP = False` — keep one map for all episodes; set `True` for a fresh layout each episode
- `POLICY_RNG` — fixed seed so the random/mixed policy choices are reproducible

In [ ]:
import os

import gymnasium as gym
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import procedural_frozenlake
from IPython.display import HTML, display

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

NUM_EPISODES = 20
MAX_STEPS = 200
FPS = 8
REGENERATE_MAP = False
POLICY_RNG = np.random.default_rng(0)

## Policy: random → Q\*

We linearly interpolate between exploration and exploitation across episodes:

| Episode | Random action probability |
|---------|---------------------------|
| 1 | 100% random |
| 10 | ~53% random |
| 20 | 0% — always `argmax(info["q_star"])` |

At each step, sample a random action with probability `random_prob`; otherwise take the greedy Q\* action from `info["q_star"]` (shape `(4,)` — one value per direction).

In [ ]:
def random_action_probability(episode: int, *, num_episodes: int = NUM_EPISODES) -> float:
    """Episode 0 → 1.0 (random). Last episode → 0.0 (Q* greedy)."""
    if num_episodes <= 1:
        return 0.0
    return 1.0 - episode / (num_episodes - 1)


def select_action(
    env,
    info: dict,
    *,
    random_prob: float,
    rng: np.random.Generator,
) -> int:
    if random_prob > 0.0 and (random_prob >= 1.0 or rng.random() < random_prob):
        return int(env.action_space.sample())
    return int(info["q_star"].argmax())

## Create the environment

Use `gym.make` with the registered id. Important kwargs:

- `render_mode="rgb_array"` — return a `(H, W, 3)` uint8 frame from `env.render()`
- `emit_q_star=True` — run value iteration on the current map and attach `info["q_star"]` each step
- `fog_of_war=True` — unvisited tiles render as `?`, including trees; bumping a tree reveals it; revealed tiles persist until the map regenerates
- `width` / `height` / `min_border` / `max_border` — fixed 10×10 canvas with tree border thickness sampled between 1 and 3 on every side
- `shoreline_jaggedness=2` — bays and peninsulas up to two tiles deep on each edge
- `mirror_prob=0.2` — sprinkle slippery mirror ice (`M`) tiles (polished-ice icons)
- `sleigh_pair_count=1` — place one pair of sleigh (`W`) warp tiles
- `start_pos_prob=0.05` / `goal_pos_prob=0.05` — place **multiple starts and goals** probabilistically; each `reset()` samples one of the start tiles
- `goal_reward_low=0.5` / `goal_reward_high=1.5` — each goal gets its own sampled reward; the present's bow renders yellow for low rewards and green for high, with the value shown in a badge on the tile
- `map_seed=16` — fixes the generated map layout (independent of `reset(seed=…)`); this seed yields 4 starts and 4 goals

The map is generated lazily on the first `reset()`. Observation space is always `Discrete(width * height)`.

In [ ]:
env = gym.make(
    procedural_frozenlake.PROCEDURAL_FROZENLAKE_ENV_ID,
    render_mode="rgb_array",
    map_seed=16,
    emit_q_star=True,
    emit_map=True,
    fog_of_war=True,
    hole_prob=0.12,
    mirror_prob=0.2,
    sleigh_pair_count=1,
    start_pos_prob=0.05,
    goal_pos_prob=0.05,
    goal_reward_low=0.5,
    goal_reward_high=1.5,
    width=10,
    height=10,
    min_border=1,
    max_border=3,
    shoreline_jaggedness=2,
)

## Text map

With `emit_map=True`, every `reset()` and `step()` puts a blueprint dict in `info["map"]` (pass it back as `fixed_map` to rebuild the same layout). The helper below prints it in a fixed-width grid with column/row labels, a border, and a tile legend so you can read the layout before watching the rollout.

In [ ]:
TILE_LEGEND = {
    "S": "Start",
    "F": "Frozen (safe ice)",
    "M": "Mirror ice (slippery)",
    "W": "Warp sleigh",
    "H": "Hole",
    "G": "Goal",
    "T": "Tree (blocked)",
}


COL_W = 5  # cell width — keeps column indices readable and aligned


def print_pretty_map(data: dict) -> None:
    """Print the board with aligned grid, indices, legend, and sleigh pairs."""
    board: list[str] = data["board"]
    height = len(board)
    width = len(board[0]) if board else 0
    row_w = max(2, len(str(height - 1)))
    left_margin = row_w + 2
    h_sep = "-" * COL_W

    def h_rule() -> str:
        return " " * left_margin + "+" + (f"{h_sep}+" * width)

    def format_row(row_idx: int, chars: str) -> str:
        cells = "|".join(f"{ch:^{COL_W}}" for ch in chars)
        return f"{row_idx:>{row_w}}  |{cells}|"

    def format_header() -> str:
        cells = " ".join(f"{c:^{COL_W}}" for c in range(width))
        return f"{' ':>{row_w}}   {cells} "

    print("MAP LAYOUT")

    print()
    print(format_header())
    print(h_rule())
    for row_idx, row in enumerate(board):
        print(format_row(row_idx, row))
        print(h_rule())

    print()
    print("Tile legend:")
    name_w = max(len(name) for name in TILE_LEGEND.values())
    for symbol, name in TILE_LEGEND.items():
        print(f"  {symbol}   {name:<{name_w}}")

    sleighs = data.get("sleighs", {}).get("pairs") or []
    if sleighs:
        print()
        print("Sleigh warp pairs:")
        for left, right in sleighs:
            lr, lc = divmod(left, width)
            rr, rc = divmod(right, width)
            print(
                f"  ({lr:>{row_w}},{lc:>{COL_W}})  state {left:>2}  <->  "
                f"({rr:>{row_w}},{rc:>{COL_W}})  state {right:>2}"
            )

    rewards = data.get("rewards") or {}
    if rewards:
        print()
        print("Goal rewards (bow renders yellow → green from low to high):")
        for state, reward in sorted(rewards.items()):
            gr, gc = divmod(int(state), width)
            print(f"  ({gr:>{row_w}},{gc:>{COL_W}})  state {int(state):>2}   reward {reward:.2f}")


_, preview_info = env.reset(seed=0)
print_pretty_map(preview_info["map"])

## Run the rollout

Standard Gymnasium loop: `reset()` then `step(action)` until `terminated` or `truncated`.

- Pass `options={"regenerate_map": True}` to `reset()` only when you want a new layout
- Call `env.render()` after each reset and step to capture frames for the video
- `reset(seed=episode)` varies episode-level randomness without changing the map (when `REGENERATE_MAP` is `False`)

Each printed line shows the episode index, current random probability, step count, and return.

In [ ]:
frames: list = []
returns: list[float] = []
reset_options = {"regenerate_map": True} if REGENERATE_MAP else None

for episode in range(NUM_EPISODES):
    random_prob = random_action_probability(episode)
    obs, info = env.reset(seed=episode, options=reset_options)
    frame = env.render()
    if frame is not None:
        frames.append(frame)

    total_reward = 0.0
    for step in range(MAX_STEPS):
        action = select_action(env, info, random_prob=random_prob, rng=POLICY_RNG)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)
        frame = env.render()
        if frame is not None:
            frames.append(frame)
        if terminated or truncated:
            returns.append(total_reward)
            policy_label = "random" if random_prob >= 1.0 else ("Q*" if random_prob <= 0.0 else f"mix p_rand={random_prob:.2f}")
            print(
                f"Episode {episode + 1:2d} ({policy_label}): "
                f"{step + 1} steps, return={total_reward:.3f}"
            )
            break
    else:
        returns.append(total_reward)
        print(f"Episode {episode + 1:2d}: hit step limit, return={total_reward:.3f}")

env.close()
print(f"\nCaptured {len(frames)} frames across {len(returns)} episodes.")
print(f"Mean return: {sum(returns) / len(returns):.3f}")

## Watch the replay

Matplotlib's `FuncAnimation` encodes the frame list as an inline HTML5 video — nothing is written to disk.

The figure is sized to the env frame with a small border (`pad = 0.02`) and no title.

In [ ]:
def display_env_replay(frames: list, *, fps: int = FPS) -> None:
    height, width = frames[0].shape[:2]
    dpi = 100
    pad = 0.02
    fig, ax = plt.subplots(figsize=(width / dpi, height / dpi), dpi=dpi)
    fig.subplots_adjust(pad, pad, 1 - pad, 1 - pad, wspace=0, hspace=0)
    ax.set_axis_off()
    ax.set_position([pad, pad, 1 - 2 * pad, 1 - 2 * pad])
    img = ax.imshow(frames[0])
    img.set_clip_on(False)

    def update(t: int):
        img.set_data(frames[t])
        return (img,)

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=1000 / fps,
        blit=True,
    )
    plt.close(fig)
    display(HTML(ani.to_html5_video()))


display_env_replay(frames)